In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional
import torch

from spice import SpiceEstimator, SpiceConfig, csv_to_dataset, BaseModel, plot_session, split_data_along_sessiondim

sys.path.append('../../..')
from weinhardt2026.utils.benchmarking_gru import training
from benchmarking_bustamante2023 import MarginalValueTheoremModel, get_dataset, generate_behavior
from spice_bustamante2023 import SpiceModel, spice_config

## Load dataset

Let's load the data first with the `csv_to_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [2]:
path_data = 'data/bustamante2023.csv'
test_sessions = (3, 6)

dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([1997, 112, 1, 11])
Number of participants: 250
Number of actions in dataset: 2
Number of additional inputs: 4


## SPICE Setup

Now we are going to define the configuration for SPICE with a `SpiceConfig` object.

The `SpiceConfig` takes as arguments 
1. `library_setup (dict)`: Defining the variable names of each module.
2. `memory_state (dict)`: Defining the memory state variables and their initial values.
3. `states_in_logit (list)`: Defining which of the memory state variables are used later for the logit computation. This is necessary for some background processes.  

And then we are going to define the SPICE model which is a child of the `BaseModel` and `torch.nn.Module` class and takes as required arguments:
1. `spice_config (SpiceConfig)`: previously defined SpiceConfig object
2. `n_actions (int)`: number of possible actions in your dataset (including non-displayed ones if applicable).
3. `n_participants (int)`: number of participants in your dataset.

As usual for a `torch.nn.Module` we have to define at least the `__init__` method and the `forward` method.
The `forward` method gets called when computing a forward pass through the model and takes as inputs `(inputs (SpiceDataset.xs), prev_state (dict, default: None), batch_first (bool, default: False))` and returns `(logits (torch.Tensor, shape: (n_participants*n_blocks*n_experiments, timesteps, n_actions)), updated_state (dict))`. Two necessary method calls inside the forward pass are:
1. `self.init_forward_pass(inputs, prev_state) -> SpiceSignals`: returns a `SpiceSignals` object which carries all relevant information already processed.
2. `self.post_forward_pass(SpiceSignals, batch_first) -> SpiceSignals`: does some re-arranging of the logits to adhere to `batch_first`.

In [ ]:
# spice_config = SpiceConfig(
#     library_setup={
#         'reward_environment': (
#             'reward[t]',
#             # 'reward[t-1]',
#             # 'reward[t-2]',
#             ),
#         'reward_patch_harvest': (
#             'reward[t]',
#             # 'reward[t-1]',
#             # 'reward[t-2]',
#             ),
#         'reward_patch_exit': (
#             # 'reward[t-1]',
#             # 'reward[t-2]',
#             ),
#         'depletion_patch_harvest': (
#             'dreward[t]', 
#             # 'dreward[t-1]',
#             # 'dreward[t-2]',
#             ),
#         'depletion_patch_exit': (
#             # 'dreward[t-1]',
#             # 'dreward[t-2]',
#             ),
#         'continuation_patch': (
#             'action[t-1]',
#             # 'action[t-2]',
#             ),
#         # 'continuation_patch_harvest': (
#         #     'action[t-1]',
#         #     # 'action[t-2]',
#         #     ),
#         # 'continuation_patch_exit': (
#         #     # 'action[t-1]',
#         #     # 'action[t-2]',
#         #     ),
#     },
#     memory_state={
#         'value_reward_environment': 0.5,
#         'value_reward_patch': 0.5,
#         'value_depletion_patch': 0,
#         'value_continuation_patch': 0,
#         'reward[t-1]': 1,
#         'action[t-1]': 0,
#     },
#     states_in_logit=(
#         'value_reward_environment',
#         'value_reward_patch',
#         'value_depletion_patch',
#         'value_continuation_patch',
#     ),
# )

In [ ]:
# class SpiceModel(BaseModel):
    
#     def __init__(self, **kwargs):
#         super().__init__(**kwargs)
        
#         self.dropout = 0.1
        
#         self.participant_embedding = self.setup_embedding(
#             num_embeddings=self.n_participants,
#             embedding_size=self.embedding_size,
#             dropout=self.dropout,
#         )
        
#         self.setup_module(key_module='reward_environment', input_size=1 + self.embedding_size, include_state=True)        
#         self.setup_module(key_module='reward_patch_harvest', input_size=1 + self.embedding_size, include_state=True)
#         self.setup_module(key_module='reward_patch_exit', input_size=self.embedding_size, include_state=True)
        
#         self.setup_module(key_module='depletion_patch_harvest', input_size=1 + self.embedding_size, include_state=True)
#         self.setup_module(key_module='depletion_patch_exit', input_size=0 + self.embedding_size, include_state=True)

#         self.setup_module(key_module='continuation_patch', input_size=1 + self.embedding_size, include_state=True)        
        
#     def forward(self, inputs, prev_state=None):
        
#         spice_signals = self.init_forward_pass(inputs, prev_state)
        
#         # used to update only the action value for harvesting. the action value for exit is kept at 0 as a reference point like in the MVT. 
#         mask_harvest_value = torch.zeros_like(spice_signals.actions[0])
#         mask_harvest_value[..., 0] = 1
        
#         action_harvest = spice_signals.actions[..., 0].unsqueeze(-1).expand_as(spice_signals.actions)
#         action_exit = spice_signals.actions[..., 1].unsqueeze(-1).expand_as(spice_signals.actions)
#         rewards = spice_signals.rewards[..., 0].unsqueeze(-1).expand_as(spice_signals.actions)
        
#         participant_embedding = self.participant_embedding(spice_signals.participant_ids)
                
#         for trial in spice_signals.trials:
            
#             harvested = action_harvest[trial] * mask_harvest_value
#             exited = action_exit[trial] * mask_harvest_value

#             # 1. Reward processing
#             # learning average reward value of environment (only information accumulation; no exit information)
#             self.call_module(
#                 key_module='reward_environment',
#                 key_state='value_reward_environment',
#                 action_mask=harvested,
#                 inputs=(
#                     rewards[trial],
#                     # self.state['reward[t-1]'],
#                     # self.state['reward[t-2]'],
#                     ),
#                 participant_index=spice_signals.participant_ids,
#                 participant_embedding=participant_embedding,
#             )
            
#             # learning current patch reward → patch valuation
#             self.call_module(
#                 key_module='reward_patch_harvest',
#                 key_state='value_reward_patch',
#                 action_mask=harvested,
#                 inputs=(
#                     rewards[trial],
#                     # self.state['reward[t-1]'],
#                     # self.state['reward[t-2]'],
#                     ),
#                 participant_index=spice_signals.participant_ids,
#                 participant_embedding=participant_embedding,
#             )
            
#             self.call_module(
#                 key_module='reward_patch_exit',
#                 key_state='value_reward_patch',
#                 action_mask=exited,
#                 inputs=None,
#                 participant_index=spice_signals.participant_ids,
#                 participant_embedding=participant_embedding,
#             )
            
            
#             # 2. Depletion processing
#             # depletion mask: only update if previous action was harvest (valid dreward)
#             # prev_was_harvest = (self.state['action[t-1]'][..., 0:1] > 0.5).float().expand_as(harvested)
#             # mask_depletion_harvest = harvested * prev_was_harvest
            
#             # learning current patch depletion → exhaustion signal
#             self.call_module(
#                 key_module='depletion_patch_harvest',
#                 key_state='value_depletion_patch',
#                 action_mask=harvested,#mask_depletion_harvest,
#                 inputs=(
#                     rewards[trial]-self.state['reward[t-1]'],
#                     ),
#                 participant_index=spice_signals.participant_ids,
#                 participant_embedding=participant_embedding,
#             )
            
#             self.call_module(
#                 key_module='depletion_patch_exit',
#                 key_state='value_depletion_patch',
#                 action_mask=exited,
#                 inputs=None,
#                 participant_index=spice_signals.participant_ids,
#                 participant_embedding=participant_embedding,
#             )
            
            
#             # 3. Continuation effect (e.g. pressure to exit to get to a fresh patch; reward/depletion agnostic)
#             self.call_module(
#                 key_module='continuation_patch',
#                 key_state='value_continuation_patch',
#                 action_mask=None,
#                 inputs=(
#                     spice_signals.actions[trial],
#                     # self.state['action[t-1]'],
#                     ),
#                 participant_index=spice_signals.participant_ids,
#                 participant_embedding=participant_embedding,
#             )
            
#             # self.call_module(
#             #     key_module='continuation_patch_exit',
#             #     key_state='value_continuation_patch',
#             #     action_mask=exited,
#             #     inputs=None,
#             #     participant_index=spice_signals.participant_ids,
#             #     participant_embedding=participant_embedding,
#             # )
            
            
#             # 4. Heuristical memory state updating
            
#             # prev_reward: store current reward if harvested, reset to -1 otherwise
#             # (only harvesting yields a reward; exiting does not)
#             self.state['reward[t-1]'] = torch.where(harvested > 0.5, rewards[trial], 1)
            
#             # prev_action: store current trial's action for use in the next trial
#             self.state['action[t-1]'] = spice_signals.actions[trial]

#             # working memory updates
#             # self.state['reward[t-2]'] = torch.where(harvested>0.5, self.state['reward[t-1]'], 0)
#             # self.state['reward[t-1]'] = torch.where(harvested>0.5, reward, 0)
            
#             # 4. Logit computation
#             spice_signals.logits[trial] = (
#                 + self.state['value_reward_environment']
#                 + self.state['value_reward_patch']
#                 + self.state['value_depletion_patch']
#                 + self.state['value_continuation_patch']
#             )
        
#         spice_signals = self.post_forward_pass(spice_signals)
        
#         return spice_signals.logits, self.get_state()

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
# fitting without SINDy coefficients

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=SpiceModel,
        spice_config=spice_config,
        n_actions=info_dataset['n_actions'],
        n_participants=info_dataset['n_participants'],
        
        epochs=0,
        warmup_steps=200,
        ensemble_size=10,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [ ]:
path_spice = 'params/spice_bustamante2023.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=SpiceModel,
        spice_config=spice_config,
        n_actions=2,
        n_participants=n_participants,
        n_experiments=1,
        
        epochs=1000,
        warmup_steps=200,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        verbose=True,
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_train.xs, dataset_train.ys)

In [ ]:
estimator.load_spice(path_spice)

In [ ]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)

## Benchmarking

### MVT Model by Constantino et al. (2015)

In [ ]:
# 1. stick to low effort 
# 2. two sets of params for low and high effort

mvt = MarginalValueTheoremModel(
    n_participants=n_participants,
    depletion=None,  # if None: learn value; else: fix to given value;
    baseline_gain=None,  # if None: learn value; else: fix to given value;
    batch_first=True,
    )

path_mvt = path_spice.replace('spice_', 'mvt_')

In [ ]:
epochs = 1000
optimizer = torch.optim.Adam(params=mvt.parameters(), lr=0.01)

mvt = training(
    model=mvt,
    optimizer=optimizer,
    epochs=epochs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    device=torch.device('cpu'),
)

torch.save(mvt.state_dict(), path_mvt)

In [ ]:
mvt.load_state_dict(torch.load(path_mvt, map_location='cpu'))

### GRU Model

In [ ]:
from weinhardt2026.utils.benchmarking_gru import GRUModel

gru = GRUModel(n_actions)

path_gru = 'params/gru_bustamante2023.pkl'

In [ ]:
epochs = 1000
optimizer = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

In [ ]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

# GENERATE BEHAVIOR

In [ ]:
generated_dataset_spice = generate_behavior(
    dataset=dataset_train,
    model=SpiceEstimator,
    save_dataset='data/bustamante2023_spice.csv'
)

generated_dataset_benchmark = generate_behavior(
    dataset=dataset_train,
    model=mvt,
    save_dataset='data/bustamante2023_benchmark.csv'
)

generated_dataset_gru = generate_behavior(
    dataset=dataset_train,
    model=gru,
    save_dataset='data/bustamante2023_gru.csv'
)

# ANALYSIS

In [ ]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals

## General analysis

In [ ]:
df = pd.read_csv(path_data)
df['last_reward'] = df.groupby('subject_id')['last_reward'].fillna(method='bfill')
df['participant_id'] = pd.factorize(df['subject_id'])[0]
exit_df = df[df['decision'] == 1]
mean_exit_threshold = exit_df.groupby(['participant_id', 'subject_id'])['last_reward'].mean().reset_index()
mean_exit_threshold['relative_optimal'] = mean_exit_threshold['last_reward'] - 6.78 #from Bustamante et al. Table S7, experiment 1
mean_exit_threshold['over_harvester'] = np.where(mean_exit_threshold['relative_optimal'] <= 0, 1, 0)
print(mean_exit_threshold)
overharvesters = mean_exit_threshold[mean_exit_threshold['over_harvester'] == 1]['participant_id'].unique()
underharvesters = mean_exit_threshold[mean_exit_threshold['over_harvester'] == 0]['participant_id'].unique()

print('OVERHARVESTERS') 
for p in overharvesters:
    print('Participant number', p)
    estimator.print_spice_model(participant_id=p)

print('UNDERHARVESTERS') 
for p in underharvesters:
    print('Participant number', p)
    estimator.print_spice_model(participant_id=p)

In [ ]:
print("Fitted MVT parameters:")
print("\nAlpha")
print(mvt.alpha_env)
print("\nBeta")
print(mvt.beta)
print("\nC")
print(mvt.c)
print("\nBaseline Gain")
print(mvt.baseline_gain)
print("\nDepletion")
print(mvt.depletion)

## Analysis model evaluation

In [ ]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=mvt.to(torch.device('cpu')),
    gru_model=gru.to(torch.device('cpu')),
)

## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
analysis_coefficients_individuals(
    criterion="SomeCriterionColumnInYourDataset",
    analysis="disc",  # also: "cont"
    reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disct"
    
    path_data=path_data,
    
    spice_model=estimator,
    
    dir_output='results'
)

## Analysis generative behavior

In [ ]:
# create violin plots